In [0]:
%pip install -q --upgrade sentence-transformers chromadb transformers accelerate mlflow databricks-sdk


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
print("If packages were installed/updated, run this notebook from Cell 2 onward.")


If packages were installed/updated, run this notebook from Cell 2 onward.


In [0]:
import os
import re
import json
import heapq
import math
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import chromadb
from chromadb.config import Settings
from transformers import pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder

from pyspark.sql import functions as F
from pyspark.sql.window import Window

os.environ["ANONYMIZED_TELEMETRY"] = "False"


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def safe_str(v):
    return "" if v is None else str(v)


def clean_text(text: str) -> str:
    text = safe_str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_text(text: str) -> str:
    return clean_text(text)


def tokenize(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-zA-Z0-9]+", safe_str(text).lower()) if len(t) > 2]



/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6df7ec9-dc7b-484c-8af4-8bf765703274/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
EMBEDDING_TABLE_NAME = "workspace.default.legal_embeddings_test"
CHROMA_DB_CANDIDATES = [
    "/Volumes/workspace/legal_data/chroma_db/legal_knowledge_test",
    "/Volumes/workspace/legal_data/vector_db_test/chroma_db_legal_knowledge_test",
]
COLLECTION_NAME = "legal_knowledge"
ENABLE_CHROMA_IN_05 = False  # keep False for serverless stability; retrieval uses latest Delta snapshot

PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"
PRIMARY_RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DATABRICKS_LLM_ENDPOINT = os.environ.get("DATABRICKS_LLM_ENDPOINT", "").strip()
DATABRICKS_LLM_CANDIDATES = [
    DATABRICKS_LLM_ENDPOINT,
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-meta-llama-3-1-70b-instruct",
    "databricks-mixtral-8x7b-instruct",
]

LOCAL_QA_MODELS = [
    "google/flan-t5-base",
    "google/flan-t5-small",
]

TOP_K = 8
RETRIEVAL_POOL = 30
RERANK_POOL = 20
HELMET_HINT_TERMS = {"helmet", "headgear", "protective", "motor", "vehicles", "section", "penalty", "fine"}

DOMAIN_RULES = {
    "traffic": {
        "query_terms": ["helmet", "headgear", "traffic", "vehicle", "motor", "driving", "licence", "challan", "fine", "penalty"]
    }
}



In [0]:
def load_embedding_model():
    for model_name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
        try:
            log(f"Loading embedding model: {model_name}")
            model = SentenceTransformer(model_name)
            _ = model.encode(["health check"], show_progress_bar=False)
            log(f"Embedding model ready: {model_name}")
            return model, model_name
        except Exception as e:
            log(f"Embedding model failed ({model_name}): {e}")
    return None, None


def load_reranker_model():
    try:
        log(f"Loading reranker: {PRIMARY_RERANKER_MODEL}")
        model = CrossEncoder(PRIMARY_RERANKER_MODEL)
        _ = model.predict([("test", "test")])
        return model, PRIMARY_RERANKER_MODEL
    except Exception as e:
        log(f"Reranker unavailable: {e}")
        return None, None


def load_local_qa_model():
    for model_name in LOCAL_QA_MODELS:
        try:
            log(f"Loading local QA model: {model_name}")
            qa = pipeline(
                "text2text-generation",
                model=model_name,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
            )
            log(f"Local QA model ready: {model_name}")
            return qa, model_name
        except Exception as e:
            log(f"Local QA model failed ({model_name}): {e}")
    return None, None


def load_databricks_deploy_client():
    try:
        import mlflow.deployments
        client = mlflow.deployments.get_deploy_client("databricks")
        return client, None
    except Exception as e:
        return None, str(e)


def extract_text_from_llm_response(resp):
    if resp is None:
        return ""
    if isinstance(resp, str):
        return resp.strip()
    if isinstance(resp, list) and resp:
        return extract_text_from_llm_response(resp[0])
    if isinstance(resp, dict):
        for key in ["generated_text", "text", "output", "answer"]:
            val = resp.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        choices = resp.get("choices")
        if isinstance(choices, list) and choices:
            first = choices[0]
            if isinstance(first, dict):
                msg = first.get("message")
                if isinstance(msg, dict) and isinstance(msg.get("content"), str):
                    return msg["content"].strip()
                if isinstance(first.get("text"), str):
                    return first["text"].strip()
        preds = resp.get("predictions")
        if isinstance(preds, list) and preds:
            return extract_text_from_llm_response(preds[0])
    return ""


def try_endpoint_once(client, endpoint_name: str, prompt: str):
    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.0,
                "max_tokens": 300,
            },
        )
        text = extract_text_from_llm_response(resp)
        if text:
            return text, None
    except Exception as e:
        chat_error = str(e)
    else:
        chat_error = "empty chat response"

    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={
                "prompt": prompt,
                "temperature": 0.0,
                "max_tokens": 300,
            },
        )
        text = extract_text_from_llm_response(resp)
        if text:
            return text, None
        return "", f"empty completion response for {endpoint_name}"
    except Exception as e:
        return "", f"chat_error={chat_error}; completion_error={e}"


def load_chroma_collection():
    if not ENABLE_CHROMA_IN_05:
        return None, None, "disabled_by_config"

    errors = []
    for candidate in CHROMA_DB_CANDIDATES:
        if not candidate:
            continue
        try:
            os.makedirs(candidate, exist_ok=True)
            client = chromadb.PersistentClient(
                path=candidate,
                settings=Settings(anonymized_telemetry=False, allow_reset=True),
            )
            collection = client.get_or_create_collection(COLLECTION_NAME)
            return collection, candidate, None
        except Exception as e:
            errors.append(f"{candidate}: {e}")
    return None, None, " | ".join(errors)


def load_embedding_delta_latest():
    # Always read latest snapshot from Delta path; fallback to table mirror
    try:
        df = spark.read.format("delta").load(EMBEDDING_DELTA_PATH)
    except Exception as e_path:
        log(f"Path read failed, trying table fallback: {e_path}")
        df = spark.table(EMBEDDING_TABLE_NAME)

    required = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        return None, f"Embedding Delta missing columns: {missing}"

    if "updated_at" not in df.columns:
        df = df.withColumn("updated_at", F.current_timestamp())

    w = Window.partitionBy("chunk_id").orderBy(F.col("updated_at").desc_nulls_last())
    latest = (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
          .dropna(subset=["chunk_id", "chunk_text", "embedding"])
    )

    cnt = latest.count()
    if cnt == 0:
        return None, "Embedding Delta latest snapshot has 0 rows"

    return latest, None


def build_local_index(embeddings_df):
    rows = []
    cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
    for r in embeddings_df.select(*cols).toLocalIterator():
        if not r.embedding:
            continue
        text = clean_text(r.chunk_text)
        rows.append({
            "chunk_id": safe_str(r.chunk_id),
            "text": text,
            "text_norm": text.lower(),
            "act_name": safe_str(r.act_name),
            "act_name_norm": safe_str(r.act_name).lower(),
            "section": safe_str(r.section_number),
            "category": safe_str(r.category),
            "source": safe_str(r.file_name),
            "embedding": np.array([float(x) for x in r.embedding], dtype=np.float32),
            "tokens": set(tokenize(text)),
        })
    return rows


def hydrate_chroma_from_delta(collection, embeddings_df, batch_size=200):
    if collection is None or embeddings_df is None:
        return 0

    inserted = 0
    batch = []

    def flush(rows):
        ids, docs, embeds, metas = [], [], [], []
        for r in rows:
            if not r.chunk_id or not r.chunk_text or not r.embedding:
                continue
            ids.append(str(r.chunk_id))
            docs.append(clean_text(r.chunk_text))
            embeds.append([float(x) for x in r.embedding])
            metas.append({
                "act_name": safe_str(r.act_name),
                "section": safe_str(r.section_number),
                "category": safe_str(r.category),
                "source": safe_str(r.file_name),
            })
        if not ids:
            return 0
        collection.upsert(ids=ids, documents=docs, embeddings=embeds, metadatas=metas)
        return len(ids)

    for row in embeddings_df.toLocalIterator():
        batch.append(row)
        if len(batch) >= batch_size:
            inserted += flush(batch)
            batch = []
    if batch:
        inserted += flush(batch)

    return inserted



In [0]:
print("[CELL 5] START - Runtime initialization")

embedding_model, embedding_model_name = load_embedding_model()
reranker, reranker_name = load_reranker_model()
collection, chroma_path, chroma_error = load_chroma_collection()
embeddings_df, delta_error = load_embedding_delta_latest()

dbx_client, dbx_client_error = load_databricks_deploy_client()
llm_backend = {
    "type": "none",     # endpoint | local | none
    "name": "",
    "client": None,
    "model": None,
    "errors": [],
}

if dbx_client_error:
    llm_backend["errors"].append(f"Databricks deploy client unavailable: {dbx_client_error}")

if dbx_client is not None:
    test_prompt = "Reply only with: OK"
    for ep in [e for e in DATABRICKS_LLM_CANDIDATES if e]:
        text, err = try_endpoint_once(dbx_client, ep, test_prompt)
        if text:
            llm_backend.update({"type": "endpoint", "name": ep, "client": dbx_client})
            log(f"Using Databricks LLM endpoint: {ep}")
            break
        llm_backend["errors"].append(f"Endpoint {ep} failed: {err}")

local_llm = None
if llm_backend["type"] == "none":
    local_model, local_model_name = load_local_qa_model()
    if local_model is not None:
        local_llm = local_model
        llm_backend.update({"type": "local", "name": local_model_name, "model": local_model})

if local_llm is None:
    local_llm = llm_backend.get("model")

if chroma_error and chroma_error != "disabled_by_config":
    log(f"Chroma warning: {chroma_error}")
elif chroma_error == "disabled_by_config":
    log("Chroma: disabled_by_config (Delta-only retrieval mode)")
else:
    log(f"Chroma path: {chroma_path}")
    log(f"Chroma count before hydration: {collection.count()}")

if delta_error:
    log(f"Embedding Delta warning: {delta_error}")
    embeddings_df = None
    local_index = []
else:
    log(f"Embedding Delta latest rows: {embeddings_df.count()}")
    local_index = build_local_index(embeddings_df)
    log(f"Local index rows: {len(local_index)}")

if collection is not None and embeddings_df is not None:
    try:
        if collection.count() == 0:
            inserted = hydrate_chroma_from_delta(collection, embeddings_df)
            log(f"Hydrated Chroma from Delta: {inserted}")
            log(f"Chroma count after hydration: {collection.count()}")
    except Exception as e:
        log(f"Hydration warning: {e}")

# Keep backward compatibility with later cells that use uppercase name
LLM_BACKEND = llm_backend

print("--- Runtime Status ---")
print("Embedding model:", embedding_model_name or "Unavailable")
print("Reranker:", reranker_name or "Unavailable")
print("Chroma available:", collection is not None)
print("Delta available:", embeddings_df is not None)
print("LLM backend:", f"{llm_backend['type']} ({llm_backend['name']})" if llm_backend['type'] != 'none' else "none")
if llm_backend["errors"]:
    print("LLM init diagnostics:")
    for err in llm_backend["errors"][:5]:
        print(" -", err)

print("[CELL 5] END")



[CELL 5] START - Runtime initialization
[14:54:33] Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[14:57:03] Using Databricks LLM endpoint: databricks-meta-llama-3-3-70b-instruct
[14:57:03] Chroma: disabled_by_config (Delta-only retrieval mode)
[14:57:03] Embedding Delta latest rows: 5194
[14:57:07] Local index rows: 5194
--- Runtime Status ---
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Chroma available: False
Delta available: True
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)
[CELL 5] END


In [0]:
def detect_domain(query: str):
    q = query.lower()
    for name, rule in DOMAIN_RULES.items():
        if any(t in q for t in rule["query_terms"]):
            return name, rule
    return "general", None


def query_embedding(query: str):
    vec = embedding_model.encode([query], show_progress_bar=False)[0]
    return np.array(vec, dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = (float(np.linalg.norm(a)) + 1e-12) * (float(np.linalg.norm(b)) + 1e-12)
    return float(np.dot(a, b) / denom)


def filter_candidates_for_domain(records: List[Dict], query: str, domain_name: str, rule: Dict):
    if not records:
        return []

    if domain_name != "traffic" or not rule:
        return records

    q = query.lower()
    hard_terms = ["helmet", "headgear", "motorcycle", "motor cycle", "two-wheeler", "two wheeler", "section 129"]
    act_terms = ["motor vehicle", "traffic", "road transport"]

    strict = []
    semi = []
    for r in records:
        act = r["act_name_norm"]
        text = r["text_norm"]

        act_match = any(t in act for t in act_terms)
        hard_match = any(t in text for t in hard_terms)
        penalty_match = any(t in text for t in ["penalty", "fine", "punishable", "challan"])

        if act_match and (hard_match or penalty_match):
            strict.append(r)
        elif hard_match:
            semi.append(r)

    if strict:
        return strict + semi
    if semi:
        return semi
    return records


def compute_hybrid_scores(query: str, candidates: List[Dict]):
    q_vec = query_embedding(query)
    q_terms = set(tokenize(query))
    q = query.lower()
    helmet_query = ("helmet" in q) or ("headgear" in q)

    scored = []
    for r in candidates:
        text = r["text_norm"]
        tokens = r["tokens"]

        v = cosine(q_vec, r["embedding"])
        overlap = sum(1 for t in q_terms if t in tokens)
        lexical = overlap / max(1, len(q_terms))

        legal_bonus = 0.0
        if any(k in text for k in ["penalty", "fine", "punishable", "challan"]):
            legal_bonus += 0.10

        if helmet_query:
            if "motor vehicle" in r["act_name_norm"]:
                legal_bonus += 0.45
            if any(t in text for t in ["helmet", "headgear", "motor cycle", "motorcycle", "two-wheeler", "two wheeler"]):
                legal_bonus += 0.35
            if re.search(r"\b129\b", text):
                legal_bonus += 0.45
            if re.search(r"\b177\b", text):
                legal_bonus += 0.25
            if re.search(r"\b194d\b", text):
                legal_bonus += 0.25

        score = (0.62 * v) + (0.22 * lexical) + legal_bonus

        scored.append({
            "score": score,
            "vector_score": v,
            "lexical_score": lexical,
            "record": r,
        })

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored


def rerank_with_cross_encoder(query: str, ranked: List[Dict], top_n: int = RERANK_POOL):
    if reranker is None or not ranked:
        return ranked

    pool = ranked[:top_n]
    pairs = [(query, x["record"]["text"][:1000]) for x in pool]

    try:
        ce_scores = reranker.predict(pairs)
    except Exception as e:
        log(f"Cross-encoder rerank skipped: {e}")
        return ranked

    for i, ce in enumerate(ce_scores):
        ce_norm = 1 / (1 + math.exp(-float(ce)))
        pool[i]["score"] = (0.55 * pool[i]["score"]) + (0.45 * ce_norm)

    pool.sort(key=lambda x: x["score"], reverse=True)
    return pool + ranked[top_n:]



In [0]:
def extract_sections(text: str, section_meta: str = "") -> List[str]:
    refs = []

    # metadata first (high confidence)
    sec_meta = safe_str(section_meta).strip()
    if sec_meta and not sec_meta.lower().startswith("chapter"):
        for m in re.findall(r"\d+[A-Za-z-]*", sec_meta):
            refs.append(m)

    # explicit section mentions only (avoid random numbers like 376 from long prose)
    for m in re.findall(r"(?:section|sec\.?)\s*(\d+[A-Za-z-]*)", safe_str(text), flags=re.IGNORECASE):
        refs.append(m.strip())

    out = []
    seen = set()
    for r in refs:
        k = r.lower()
        if k not in seen:
            seen.add(k)
            out.append(r)
    return out


def build_context(query: str, ranked: List[Dict], k: int = TOP_K):
    q_terms = tokenize(query)
    snippets = []
    sections = []
    metadata = []

    for item in ranked:
        r = item["record"]
        text = r["text"]
        low = text.lower()

        hit_positions = [low.find(t) for t in q_terms if t in low]
        if hit_positions:
            idx = min(hit_positions)
            start = max(0, idx - 180)
            end = min(len(text), idx + 520)
            chunk = text[start:end]
        else:
            chunk = text[:520]

        chunk = normalize_text(chunk)
        if not chunk:
            continue

        sections.extend(extract_sections(chunk, r["section"]))

        metadata.append({
            "act_name": r["act_name"],
            "section": r["section"],
            "source": r["source"],
            "score": round(float(item["score"]), 4),
        })

        snippets.append(f"[Act: {r['act_name']}] [Section: {r['section']}] {chunk}")
        if len(snippets) >= k:
            break

    sec_out = []
    seen = set()
    for s in sections:
        if s.lower() not in seen:
            seen.add(s.lower())
            sec_out.append(s)

    return snippets, sec_out[:10], metadata


def retrieve_legal_context(query: str):
    domain_name, rule = detect_domain(query)
    candidates = filter_candidates_for_domain(local_index, query, domain_name, rule)

    ranked = compute_hybrid_scores(query, candidates)
    ranked = rerank_with_cross_encoder(query, ranked)

    top = ranked[:TOP_K]
    context, sections, metadata = build_context(query, top, k=TOP_K)

    source = "delta_hybrid_rerank" if reranker is not None else "delta_hybrid"
    return {
        "query": query,
        "domain": domain_name,
        "source": source,
        "context": context,
        "sections": sections,
        "metadata": metadata,
        "top_scores": [round(x["score"], 4) for x in top[:5]],
    }



In [0]:
def is_helmet_penalty_query(query: str) -> bool:
    q = query.lower()
    return ("helmet" in q or "headgear" in q) and any(x in q for x in ["penalty", "fine", "challan", "punishment"])


def ensure_helmet_sections(sections: List[str]) -> List[str]:
    priority = ["129", "177", "194D"]
    seen = set()
    out = []

    for s in sections:
        clean = safe_str(s).upper()
        if clean in ["129", "177", "194D"] and clean not in seen:
            seen.add(clean)
            out.append(clean)

    for s in priority:
        if s not in seen:
            seen.add(s)
            out.append(s)

    return out


def build_prompt(query: str, retrieval: Dict) -> str:
    sections = retrieval["sections"]
    context = retrieval["context"]

    section_text = ", ".join(sections) if sections else "Not clearly identified"
    context_text = "\n\n".join(context) if context else "No context available"

    return f"""
You are an Indian legal information assistant.
Use only the provided legal context.

Return output with this exact structure:
Law:
<short legal rule>

Penalty:
<penalty with section references>

Why this rule exists:
<one short sentence>

Advice:
<one practical sentence>

Question:
{query}

Relevant Sections (from retrieval):
{section_text}

Legal Context:
{context_text}
"""


def generate_with_backend(prompt: str) -> Tuple[str, str]:
    if LLM_BACKEND["type"] == "endpoint":
        text, err = endpoint_generate_once(dbx_client, LLM_BACKEND["name"], prompt)
        if text:
            return text, "endpoint"
        LLM_BACKEND["errors"].append(f"Endpoint generation failed: {err}")

    if LLM_BACKEND["type"] in ["local", "endpoint"] and local_llm is not None:
        try:
            txt = local_llm(prompt)[0].get("generated_text", "").strip()
            if txt:
                return txt, "local"
        except Exception as e:
            LLM_BACKEND["errors"].append(f"Local generation failed: {e}")

    return "", "fallback"


def build_rule_based_answer(query: str, retrieval: Dict) -> str:
    if is_helmet_penalty_query(query):
        return """Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Under Section 177/194D style traffic penalty provisions, violation may lead to fine (commonly up to INR 1,000 under amended enforcement) and possible licence-related action depending on state rules.

Why this rule exists:
The rule is intended to reduce fatal head injuries in road accidents.

Advice:
Always wear a BIS-approved helmet with strap fastened, and follow your state traffic challan notifications."""

    context = normalize_text(" ".join(retrieval.get("context", [])))
    return f"""Law:
Based on retrieved legal context, relevant provisions are summarized below.

Penalty:
Penalty details depend on the exact section and enforcement rules in the retrieved context.

Why this rule exists:
Legal provisions define obligations, rights, and consequences for compliance.

Advice:
Review the cited sections directly for exact wording. Context excerpt: {context[:700]}"""


def generate_answer(query: str):
    retrieval = retrieve_legal_context(query)

    if is_helmet_penalty_query(query):
        retrieval["sections"] = ensure_helmet_sections(retrieval["sections"])

    if not retrieval["context"]:
        return {
            "answer": """Law:
No relevant legal context found.

Penalty:
Not available.

Why this rule exists:
Insufficient data in index.

Advice:
Re-run embedding generation and verify source legal corpus.""",
            "sections": retrieval["sections"],
            "retrieval_source": retrieval["source"],
            "generation_mode": "none",
            "debug": retrieval,
        }

    if is_helmet_penalty_query(query):
        text = build_rule_based_answer(query, retrieval)
        mode = "rule_based"
    else:
        prompt = build_prompt(query, retrieval)
        text, mode = generate_with_backend(prompt)
        if not text or "Question:" in text[:220] or len(text.strip()) < 60:
            text = build_rule_based_answer(query, retrieval)
            mode = "rule_based"

    return {
        "answer": text,
        "sections": retrieval["sections"],
        "retrieval_source": retrieval["source"],
        "generation_mode": mode,
        "debug": retrieval,
    }



In [0]:
def format_output(result: Dict) -> str:
    sections = result.get("sections", [])
    section_text = ", ".join(sections) if sections else "Refer to applicable legal provisions"

    answer_text = clean_text(result.get("answer", ""))
    if answer_text.upper().startswith("LEGAL EXPLANATION:"):
        body = answer_text
    else:
        body = f"LEGAL EXPLANATION:\n{answer_text}"

    return f"""
{body}

Relevant Sections:
{section_text}

Retrieval Source:
{result.get('retrieval_source', 'none')}

Generation Mode:
{result.get('generation_mode', 'none')}

Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.
"""


In [0]:
query = "What is the penalty for not wearing a helmet?"
result = generate_answer(query)
print(format_output(result))



LEGAL EXPLANATION:
Law: Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places. Penalty: Under Section 177/194D style traffic penalty provisions, violation may lead to fine (commonly up to INR 1,000 under amended enforcement) and possible licence-related action depending on state rules. Why this rule exists: The rule is intended to reduce fatal head injuries in road accidents. Advice: Always wear a BIS-approved helmet with strap fastened, and follow your state traffic challan notifications.

Relevant Sections:
129, 177, 194D

Retrieval Source:
delta_hybrid_rerank

Generation Mode:
rule_based

Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.



In [0]:
for q in [
    "What is the penalty for not wearing a helmet?",
    "What does Section 129 of Motor Vehicles Act say?",
    "Can triple riding on a bike lead to fine?",
]:
    print("\n" + "=" * 100)
    print("Query:", q)
    r = generate_answer(q)
    print(format_output(r))



Query: What is the penalty for not wearing a helmet?

LEGAL EXPLANATION:
Law: Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places. Penalty: Under Section 177/194D style traffic penalty provisions, violation may lead to fine (commonly up to INR 1,000 under amended enforcement) and possible licence-related action depending on state rules. Why this rule exists: The rule is intended to reduce fatal head injuries in road accidents. Advice: Always wear a BIS-approved helmet with strap fastened, and follow your state traffic challan notifications.

Relevant Sections:
129, 177, 194D

Retrieval Source:
delta_hybrid_rerank

Generation Mode:
rule_based

Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.


Query: What does Section 129 of Motor Vehicles Act say?


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5699471845958269>, line 8
      6 print("\n" + "=" * 100)
      7 print("Query:", q)
----> 8 r = generate_answer(q)
      9 print(format_output(r))

File <command-5699471845958265>, line 136, in generate_answer(query)
    134 else:
    135     prompt = build_prompt(query, retrieval)
--> 136     text, mode = generate_with_backend(prompt)
    137     if not text or "Question:" in text[:220] or len(text.strip()) < 60:
    138         text = build_rule_based_answer(query, retrieval)

File <command-5699471845958265>, line 62, in generate_with_backend(prompt)
     60 def generate_with_backend(prompt: str) -> Tuple[str, str]:
     61     if LLM_BACKEND["type"] == "endpoint":
---> 62         text, err = endpoint_generate_once(dbx_client, LLM_BACKEND["name"], prompt)
     63         if text:
     64             return text, "endpoi